# Задача 1 – Compare

## Код

In [1]:
from time import perf_counter

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    return result + left[i:] + right[j:]


def mergesort(nums):
    if len(nums) < 2:
        return nums.copy()
    mid = len(nums) // 2
    return merge(mergesort(nums[:mid]), mergesort(nums[mid:]))


def quicksort(nums):
    if len(nums) < 2:
        return nums.copy()
    pivot = nums[0]
    left, equal, right = [], [], []
    for val in nums:
        if val < pivot:
            left.append(val)
        elif val > pivot:
            right.append(val)
        else:
            equal.append(val)
    return quicksort(left) + equal + quicksort(right)


def timed(func):
    def measure(nums):
        start = perf_counter()
        result = func(nums)
        return result, perf_counter() - start

    return measure

## Идея

В mergesort делим массив пополам и рекурсивно сортируем обе половины. В merge каждый раз берем меньший из двух первых оставшихся элементов. Когда одна половина закончилась, дописываем остаток другой.

В quicksort берем первый элемент за pivot и раскладываем значения на меньшие, равные и большие. Равные уже на своем месте между двумя группами, поэтому рекурсивно сортируем только left и right. При длине меньше 2 сортировать больше нечего.

Декоратором timed оборачиваем только внешний вызов. Замеряем время всей сортировки через perf_counter и возвращаем результат и время в секундах.

**Mergesort:** время O(n log n), тк на каждом из O(log n) уровней сливаем суммарно n элементов. Память O(n) на срезы и слияние, стек O(log n).

**Quicksort:** на перемешанных различных значениях время в среднем O(n log n), а на отсортированном входе O(n²), тк каждый раз остается часть на один элемент короче. При примерно равных частях память O(n), в худшем случае O(n²), тк массивы длин n, n − 1, … остаются в стеке до возврата. На длинном отсортированном массиве можно упереться в лимит рекурсии Python.

## Визуализация

![Схема](schema.png)

## Тесты

In [2]:
def test_sorts():
    for sort in (mergesort, quicksort):
        for nums in ([], [7], [4, 1, 3, 2, 1], [3, -2, 0, -2, 9, 1, -5],
                     [2, 2, 2], list(range(10, 0, -1))):
            original = nums.copy()
            assert sort(nums) == sorted(original)
            assert nums == original

In [3]:
test_sorts()

### Сравнение времени

Берем по 600 разных чисел в разном порядке. На отсортированном и обратном входе первый элемент всегда крайний, поэтому quicksort делит массив не поровну. Перемешанный вход нужен для сравнения. Берем медиану 7 замеров, входы готовим заранее.

In [4]:
from random import Random
from statistics import median

n = 600
ordered = list(range(n))
shuffled = ordered.copy()
Random(42).shuffle(shuffled)
measure_merge = timed(mergesort)
measure_quick = timed(quicksort)

for name, nums in [('по возрастанию', ordered),
                   ('по убыванию', ordered[::-1]),
                   ('перемешанный', shuffled)]:
    expected = sorted(nums)
    merge_times, quick_times = [], []
    for _ in range(7):
        result, elapsed = measure_merge(nums)
        assert result == expected
        merge_times.append(elapsed)
        result, elapsed = measure_quick(nums)
        assert result == expected
        quick_times.append(elapsed)
    m, q = median(merge_times), median(quick_times)
    print(f'{name}: merge {m * 1000:.2f} мс, quick {q * 1000:.2f} мс, quick/merge {q / m:.1f}')

по возрастанию: merge 0.44 мс, quick 6.26 мс, quick/merge 14.2
по убыванию: merge 0.35 мс, quick 3.33 мс, quick/merge 9.5
перемешанный: merge 0.50 мс, quick 0.26 мс, quick/merge 0.5
